<div style='text-align: center'>
    <p style='font-size:50px; margin-bottom: 0px'>Statistics</p>
    <o style='font-size:20px;'>This notebook handles calculating stats for all variations and putting them into a table</p>
</div>

In [158]:
import cogsworth
import numpy as np
import pandas as pd

import sys
sys.path.append('../src')
import plotting, helpers
from importlib import reload

In [184]:
files = [
    'fiducial',
    # singles
    'singles',
    # time-evolving potential
    # ---
    # kicks
    'bhflag_3', 'kickflag_1',
    # remnant mass prescriptions
    'fryer_rapid', 'mandel_muller', 'maltsev_fallback_0.5',
    'maltsev_fallback_0.0', 'maltsev_fallback_0.25', 'maltsev_fallback_0.75', 'maltsev_fallback_1.0',
    'maltsev_pf_prob_0.0', 'maltsev_pf_prob_1.0',
    # mt physics
    # 'beta_0.0', 'beta_0.5', 'beta_1.0',
    # 'alpha_0.1', 'alpha_0.5', 'alpha_2.0', 'alpha_10.0',
    # 'qcrit_caseB_0.001', 'qcrit_caseB_1000',
]

labels = [
    r"$f_{\rm bin} = 1.0$",
    r"$f_{\rm bin} = 0.0$",
    "No fallback rescaling", "H25",
    "F12 - Rapid", "MM20", "M25 (Default)",
    r"$f_{\rm fb} = 0.0$", r"$f_{\rm fb} = 0.25$", r"$f_{\rm fb} = 0.75$", r"$f_{\rm fb} = 1.0$",
    r"$P({\rm pf}) = 0.0$", r"$P({\rm pf}) = 1.0$",
    # r"$\beta = 0.0$", r"$\beta = 0.5$", r"$\beta = 1.0$",
    # r"$\alpha_{\rm CE} = 0.1$", r"$\alpha_{\rm CE} = 0.5$", r"$\alpha_{\rm CE} = 2.0$", r"$\alpha_{\rm CE} = 10.0$",
    # r"$q_{\rm crit, B} = 0$", r"$q_{\rm crit, B} = \infty$",
]

paper_replacers = {
    "H25": r"\citet{Hobbs+2005}",
    "F12": r"\citet{Fryer+2012:2012ApJ...749...91F}",
    "MM20": r"\citet{Mandel+2020:2020MNRAS.499.3214M}",
    "M25": r"\citet{Maltsev+2025:2025AA...700A..20M}",
}

In [185]:
pops, data = helpers.load_postprocessed_pops(files, labels, [None] * len(files))

In [186]:
reload(plotting)

scale_heights = []
scale_height_errs = []

for label in labels:
    sh, sh_err, _, _ = plotting.estimate_scale_height(
        z=np.abs(data[label]["pos"]["BH"][:, 2],),
        bins=np.linspace(0, 3, 201),
        plot=False,
        xlim=(0, 3),
        n_components=2
    )
    scale_heights.append(sh * 1000)
    scale_height_errs.append(sh_err * 1000)

In [208]:
stats = {
    "scale_up": [6e10 / data[label]["mass_binaries"] for label in labels],
    "n_BH": [len(data[label]["mass"]["BH"]) for label in labels],
    "m_BH": [sum(data[label]["mass"]["BH"]) for label in labels],
    "scale_height": scale_heights,
    "scale_height_err": scale_height_errs,
    r"$f_{\rm esc}$": [data[label]["escaped"]["BH"].sum() / len(data[label]["escaped"]["BH"])
                       for label in labels],
    r"$\langle M_{\rm BH} \rangle_{|z| < 1 \, {\rm kpc}}$": [data[label]["mass"]["BH"][(abs(data[label]["pos"]["BH"][:, 2]) < 1) & ~(data[label]["escaped"]["BH"])].mean() for label in labels],
    r"$\langle M_{\rm BH} \rangle_{|z| \ge 1 \, {\rm kpc}}$": [data[label]["mass"]["BH"][(abs(data[label]["pos"]["BH"][:, 2]) >= 1) & ~(data[label]["escaped"]["BH"])].mean() for label in labels],
}
df = pd.DataFrame(stats)
df.index = labels

df["n_BH"] *= df["scale_up"]
df["m_BH"] *= df["scale_up"]
del df["scale_up"]

df["n_BH"] /= 1e8
df["m_BH"] /= 1e9

latex_header = [r"$N_{\rm BH} / 10^8$", r"$M_{\rm BH} / 10^9 M_\odot$", r"$h_z$ (pc)", r"$\sigma_{h_z}$ (pc)"]
df.rename(columns=dict(zip(df.columns, latex_header)), inplace=True)

df

,$N_{\rm BH} / 10^8$,$M_{\rm BH} / 10^9 M_\odot$,$h_z$ (pc),$\sigma_{h_z}$ (pc),$f_{\rm esc}$,"$\langle M_{\rm BH} \rangle_{|z| < 1 \, {\rm kpc}}$","$\langle M_{\rm BH} \rangle_{|z| \ge 1 \, {\rm kpc}}$"
$f_{\rm bin} = 1.0$,1.867232,1.709923,549.549550,16.526307,0.028944,10.024751,7.683696
$f_{\rm bin} = 0.0$,1.689458,1.465573,633.633634,16.609590,0.035239,9.617527,7.432681
No fallback rescaling,1.646123,1.369075,630.630631,26.818895,0.161201,7.601871,8.687165
H25,1.646002,1.367844,606.606607,21.142039,0.032968,9.342824,7.209527
F12 - Rapid,1.226423,1.348490,468.468468,21.925983,0.008745,11.270059,10.059127
MM20,2.083642,1.719326,390.390390,22.693749,0.000106,8.341277,7.413620
M25 (Default),0.314405,0.334589,450.450450,46.821141,0.007036,11.010209,8.920383
$f_{\rm fb} = 0.0$,0.244891,0.290427,429.429429,59.368257,0.000427,11.750435,13.144621
$f_{\rm fb} = 0.25$,0.303085,0.312324,444.444444,44.512871,0.017176,10.977372,7.475113
$f_{\rm fb} = 0.75$,0.314796,0.353830,453.453453,43.330982,0.000927,11.284857,10.952853


In [200]:
np.sum(data[labels[0]]["mass"]["BH"][data[labels[0]]["escaped"]["BH"]]) * scale_up[0] / 1e7

np.float64(2.799176762622803)

In [206]:
(2.8e7 * u.Msun / (12 * u.Gyr)).to(u.Msun / (440 * u.yr))

<Quantity 1.02666667 0.00227273 solMass / yr>

In [195]:
np.percentile(data[labels[0]]["mass"]["BH"], [50, 97.5])

array([ 7.47529101, 25.65512617])

In [209]:
first_label_to_section = {
    r"$f_{\rm bin} = 1.0$": "Fiducial", "No fallback rescaling": "Supernova natal kicks",
    "F12 Rapid": "Remnant mass prescriptions",
    r"$f_{\rm fb} = 0.0$": "M25 fallback mass fraction",
    r"$P({\rm pf}) = 0.0$": "M25 partial fallback probability",
    r"$\beta = 0.0$": "Mass transfer physics"
}

cols = [col for col in df.columns if col != r"$\sigma_{h_z}$ (pc)"]
latex_table = r"""\begin{tabular}{lcccccc}
\toprule
"""
latex_table += "Model & " + " & ".join(cols) + r" \\" + "\n"
latex_table += r"\midrule" + "\n"

for i, row in df.iterrows():
    if i in first_label_to_section:
        # add a small bit of vertical space before the section header
        latex_table += r"\addlinespace[0.5em]" + "\n"
        latex_table += r"\multicolumn{7}{l}{" + r"\textbf{" + first_label_to_section[i] + r"}} \\" + "\n"
    vals = row.values
    latex_table += r"\quad " + i + f' & ${vals[0]:.2f}$ & ${vals[1]:.2f}$ & ${vals[2]:.0f} \pm {vals[3]:.0f}$ & {vals[4] * 100:1.2f}\% & ${vals[5]:.2f}$ & ${vals[6]:.2f}$ \\\\' + "\n"
latex_table += r"\bottomrule" + "\n"
latex_table += r"\end{tabular}"

for key, val in paper_replacers.items():
    latex_table = latex_table.replace(key, val)

# indent every line by 8 spaces
# latex_table = "\n".join([" " * 8 + line for line in latex_table.splitlines()])

print(latex_table)

\begin{tabular}{lcccccc}
\toprule
Model & $N_{\rm BH} / 10^8$ & $M_{\rm BH} / 10^9 M_\odot$ & $h_z$ (pc) & $f_{\rm esc}$ & $\langle M_{\rm BH} \rangle_{|z| < 1 \, {\rm kpc}}$ & $\langle M_{\rm BH} \rangle_{|z| \ge 1 \, {\rm kpc}}$ \\
\midrule
\addlinespace[0.5em]
\multicolumn{7}{l}{\textbf{Fiducial}} \\
\quad $f_{\rm bin} = 1.0$ & $1.87$ & $1.71$ & $550 \pm 17$ & 2.89\% & $10.02$ & $7.68$ \\
\quad $f_{\rm bin} = 0.0$ & $1.69$ & $1.47$ & $634 \pm 17$ & 3.52\% & $9.62$ & $7.43$ \\
\addlinespace[0.5em]
\multicolumn{7}{l}{\textbf{Supernova natal kicks}} \\
\quad No fallback rescaling & $1.65$ & $1.37$ & $631 \pm 27$ & 16.12\% & $7.60$ & $8.69$ \\
\quad \citet{Hobbs+2005} & $1.65$ & $1.37$ & $607 \pm 21$ & 3.30\% & $9.34$ & $7.21$ \\
\quad \citet{Fryer+2012:2012ApJ...749...91F} - Rapid & $1.23$ & $1.35$ & $468 \pm 22$ & 0.87\% & $11.27$ & $10.06$ \\
\quad \citet{Mandel+2020:2020MNRAS.499.3214M} & $2.08$ & $1.72$ & $390 \pm 23$ & 0.01\% & $8.34$ & $7.41$ \\
\quad \citet{Maltsev+2025:2025AA..